# NumPilot 比赛主演示：全新 Ascend 环境

这份 Notebook 验证 ordinary 主链：固定来源安装 → runtime 身份 → configured model → 静态分析 → 迁移 → 数值对拍 → strict-NPU resident benchmark。运行产物与私有配置均留在仓库外。

> 当前为 `0.3.0a1.dev0` 发布分支快照；最终提交必须切到已验证的不可变 tag，并在干净 checkout 重跑。

## 1. 显式安装开关

默认只打印命令，不联网。ordinary direct lane 不需要 CANNBot；仅在明确进入 GAP lane、阅读固定版本许可证并接受后，才同时打开两个 CANNBot 开关。

In [ ]:
from pathlib import Path
import subprocess

REPO = Path.cwd().resolve()
PERSISTENT_ROOT = Path('/absolute/persistent/numpilot-runtime')
ALLOW_NETWORK = False
WITH_CANNBOT = False
ACCEPT_CANNBOT_LICENSE = False

install = [str(REPO / 'install.sh'), '--allow-network', '--persistent-root', str(PERSISTENT_ROOT)]
if WITH_CANNBOT:
    if not ACCEPT_CANNBOT_LICENSE:
        raise RuntimeError('启用 CANNBot 前必须先审阅并接受固定版本许可证')
    install += ['--with-cannbot', '--accept-cannbot-license']
print(' '.join(install))
if ALLOW_NETWORK:
    subprocess.run(install, cwd=REPO, check=True)

## 2. Runtime 身份门

先 source 安装器打印的 activation 脚本，再执行以下检查。安装成功不等于任意 NumPy API 都在 NPU 上。

In [ ]:
checks = [
    ['python', '-m', 'numpilot', '--version'],
    ['python', '-m', 'numpilot', 'paths', '--json'],
    ['python', '-m', 'numpilot', 'bootstrap', '--verify'],
    ['python', '-m', 'numpilot', 'doctor', '--backend', 'npu'],
]
for command in checks:
    subprocess.run(command, cwd=REPO, check=True)

## 3. Configured model

运行 `python -m numpilot configure-llm --init`，在命令打印的私有文件中填写 endpoint、Token、精确模型名、provider 审计标签和 `protocol=openai-chat-completions`，再执行 `--check` 与 `--probe`。不要把 Token、私有 endpoint、代理或响应正文放进 Notebook。

In [ ]:
for action in ('--check', '--probe'):
    subprocess.run(
        ['python', '-m', 'numpilot', 'configure-llm', action],
        cwd=REPO,
        check=True,
    )

## 4. 3–5 分钟主演示

默认案例是无 `matmul` 的 `float32` Poisson-Ritz 应用阶段。只有用户明确要求“开始迁移”并确认模型配置后才打开执行开关。主结果是设备驻留重复计算；冷启动、输入准备和最终 D2H 分开报告。

In [ ]:
RUN_DEMO = False
demo = [str(REPO / 'run_demo.sh'), '--quick']
print(' '.join(demo))
if RUN_DEMO:
    subprocess.run(demo, cwd=REPO, check=True)

## 5. 发布前复核

只有 `performance_valid=true` 才能发布 speedup。最终提交还必须通过发布树门禁、证据 SHA-256、完整测试和干净工作树检查；任一身份、数值或 provenance 门失败都应停止。

In [ ]:
release_checks = [
    ['python', 'tools/check_release_tree.py'],
    ['sha256sum', '-c', 'evidence/MANIFEST.sha256'],
    ['python', '-m', 'pytest', '-q', '-p', 'no:cacheprovider'],
]
for command in release_checks:
    print(' '.join(command))